<a href="https://colab.research.google.com/github/sxsaa/Mathematical-Reasoning-in-Small-Language-Models-using-Process-Reward-Models-and-Best-of-N-Search/blob/main/Preprocess_the_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl wandb

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer
from datetime import datetime
from datasets import DatasetDict
import os
import wandb
from google.colab import userdata
from trl import SFTConfig, SFTTrainer

In [ ]:
from huggingface_hub import login

# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Code using ChatGpt

from datasets import load_dataset, Dataset
from tqdm import tqdm

print("Loading PRM800K in streaming mode...")
dataset = load_dataset(
    "tasksource/PRM800K",
    split="train",
    streaming=True
)

processed_rows = []

for example in dataset:

    steps = example.get("label", {}).get("steps")

    if not steps:
        continue

    problem_text = example.get("question", {}).get("problem")
    if not problem_text:
        continue

    context_steps = []

    for step_data in steps:

        if not step_data:
            continue

        completions = step_data.get("completions")
        if not completions:
            continue

        for candidate in completions:

            if not candidate:
                continue

            rating = candidate.get("rating")
            if rating is None:
                continue

            label = {1: "Correct", 0: "Neutral", -1: "Incorrect"}.get(rating, "Neutral")

            context_str = (
                "No previous steps."
                if not context_steps
                else "\n".join(
                    [f"Step {i+1}: {s}" for i, s in enumerate(context_steps)]
                )
            )

            prompt = (
                f"Problem: {problem_text}\n\n"
                f"History of Previous Steps:\n{context_str}\n\n"
                f"Current Step to Evaluate: {candidate.get('text','')}\n\n"
                f"Question: Label this step as Correct, Neutral, or Incorrect."
            )

            processed_rows.append({
                "messages": [
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": label}
                ]
            })

        # Update context
        human_c = step_data.get("human_completion")
        chosen_idx = step_data.get("chosen_completion")

        if isinstance(human_c, dict):
            context_steps.append(human_c.get("text", ""))
        elif (
            chosen_idx is not None
            and completions
            and chosen_idx < len(completions)
        ):
            context_steps.append(
                completions[chosen_idx].get("text", "")
            )
        else:
            break

print("Converting to HuggingFace Dataset...")
processed_dataset = Dataset.from_list(processed_rows)

print("Total rows:", len(processed_dataset))

In [ ]:
print("Total processed rows:", len(processed_rows))

sample = processed_rows[0]

for msg in sample["messages"]:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"])
    print("-" * 80)

In [ ]:
# Code Gemeni AI

from datasets import load_dataset
import json

# 1. Load in streaming mode (bypasses the casting error)
print("Loading PRM800K in streaming mode...")
ds = load_dataset("tasksource/PRM800K", split="train", streaming=True)

def process_stream(streaming_ds):
    """
    A generator that yields processed ChatML rows one by one.
    """
    for example in streaming_ds:
        # Safely get problem text and steps list
        problem_text = example.get('question', {}).get('problem')
        steps_list = example.get('label', {}).get('steps')

        if not problem_text or not steps_list:
            continue

        current_context_steps = []

        for step_data in steps_list:
            if not step_data:
                continue

            completions = step_data.get('completions')

            # Only iterate through candidates if completions exist and are not empty
            if completions:
                for candidate in completions:
                    if not candidate:
                        continue

                    rating = candidate.get('rating')
                    if rating is None:
                        continue

                    label = {1: "Correct", 0: "Neutral", -1: "Incorrect"}.get(rating, "Neutral")

                    context_str = "No previous steps." if not current_context_steps else \
                        "\n".join([f"Step {j+1}: {s}" for j, s in enumerate(current_context_steps)])

                    prompt = (
                        f"Problem: {problem_text}\n\n"
                        f"History of Previous Steps:\n{context_str}\n\n"
                        f"Current Step to Evaluate: {candidate.get('text', '')}\n\n"
                        f"Question: Label this step as Correct, Neutral, or Incorrect."
                    )

                    yield {
                        "messages": [
                            {"role": "user", "content": prompt},
                            {"role": "assistant", "content": label}
                        ]
                    }

            # Context building for the next step in the sequence
            human_c = step_data.get('human_completion')
            chosen_idx = step_data.get('chosen_completion')

            if isinstance(human_c, dict): # Check if human_c is a dict (as seen in dataset structure)
                current_context_steps.append(human_c.get('text', ''))
            elif chosen_idx is not None and completions and chosen_idx < len(completions): # Ensure completions is not None before indexing
                current_context_steps.append(completions[chosen_idx].get('text', ''))
            else:
                # If neither exists or chosen_idx is out of bounds, break the trajectory
                break

# 2. Process and save to a local JSONL file
# Since it's a stream, we can't use .map() easily for saving.
# We'll write to a file directly.
output_file = "prm800k_processed.jsonl"
print(f"Processing and saving to {output_file}...")

count = 0
with open(output_file, "w") as f:
    for processed_row in process_stream(ds):
        f.write(json.dumps(processed_row) + "\n")
        count += 1
        if count % 10000 == 0:
            print(f"Processed {count} steps...")

print(f"✅ Done! Saved {count} training examples to {output_file}")


In [ ]:
from datasets import load_dataset

# 1. Load the local JSONL file
print("Loading local JSONL file...")
dataset = load_dataset("json", data_files="prm800k_processed.jsonl", split="train")

# 2. Push to Hugging Face Hub
# Your dataset will be available at: https://huggingface.co/datasets/sxsaa/PRM800K-Step-Verifier
hf_repo_id = "sxsaa/PRM800K-Step-Verifier"

print(f"Pushing to Hugging Face at {hf_repo_id}...")
dataset.push_to_hub(hf_repo_id)

print("✅ Success! Your dataset is now live.")

In [ ]:
from datasets import load_dataset, Dataset
import json

# 1. Load the TEST split in streaming mode
print("Loading PRM800K TEST split in streaming mode...")
test_ds = load_dataset("tasksource/PRM800K", split="test", streaming=True)

def process_stream(streaming_ds):
    """
    A generator that yields processed ChatML rows one by one.
    This is identical to the logic you used for the train set.
    """
    for example in streaming_ds:
        # Safely get problem text and steps list
        problem_text = example.get('question', {}).get('problem')
        steps_list = example.get('label', {}).get('steps')

        if not problem_text or not steps_list:
            continue

        current_context_steps = []

        for step_data in steps_list:
            if not step_data:
                continue

            completions = step_data.get('completions')

            # Only iterate through candidates if completions exist and are not empty
            if completions:
                for candidate in completions:
                    if not candidate:
                        continue

                    rating = candidate.get('rating')
                    if rating is None:
                        continue

                    label = {1: "Correct", 0: "Neutral", -1: "Incorrect"}.get(rating, "Neutral")

                    context_str = "No previous steps." if not current_context_steps else \
                        "\n".join([f"Step {j+1}: {s}" for j, s in enumerate(current_context_steps)])

                    prompt = (
                        f"Problem: {problem_text}\n\n"
                        f"History of Previous Steps:\n{context_str}\n\n"
                        f"Current Step to Evaluate: {candidate.get('text', '')}\n\n"
                        f"Question: Label this step as Correct, Neutral, or Incorrect."
                    )

                    yield {
                        "messages": [
                            {"role": "user", "content": prompt},
                            {"role": "assistant", "content": label}
                        ]
                    }

            # Context building for the next step in the sequence
            human_c = step_data.get('human_completion')
            chosen_idx = step_data.get('chosen_completion')

            if isinstance(human_c, dict): # Check if human_c is a dict (as seen in dataset structure)
                current_context_steps.append(human_c.get('text', ''))
            elif chosen_idx is not None and completions and chosen_idx < len(completions): # Ensure completions is not None before indexing
                current_context_steps.append(completions[chosen_idx].get('text', ''))
            else:
                # If neither exists or chosen_idx is out of bounds, break the trajectory
                break

# 2. Process the stream and collect it into a list
print("Extracting and formatting test data from the stream...")
formatted_test_data = []
count = 0

for processed_row in process_stream(test_ds):
    formatted_test_data.append(processed_row)
    count += 1
    if count % 1000 == 0:
        print(f"Processed {count} test steps...")

print(f"Total test steps extracted: {count}")

# 3. Convert to a Hugging Face Dataset
print("Converting to Hugging Face Dataset format...")
import pandas as pd
test_df = pd.DataFrame(formatted_test_data)
final_test_hf_dataset = Dataset.from_pandas(test_df)

# 4. Push directly to your existing repository as the 'test' split
target_repo_id = "sxsaa/PRM800K-Step-Verifier"

print(f"Pushing formatted dataset to the 'test' split of {target_repo_id}...")
# The split="test" argument tells Hugging Face to add this as a new tab, not overwrite 'train'
final_test_hf_dataset.push_to_hub(target_repo_id, split="test")

print("✅ Official Test split successfully added to your repository!")